In [1]:
# Druga metoda - osobne porównywanie obszaru twarzy i obszaru wokół twarzy
import cv2
import numpy as np
import cv2.data

In [2]:
# Funkcja analizująca oświetlenie twarzy na zdjęciu wejściowym.
def analyze_lighting(image, face, flash_image, flash_image_face) -> (str, bool, float):
    # Podejście: wycięcie fragmentu zdjęcia (nie całego), przeskalowanie, przekonwertowanie na CIELUV i porównanie
    # [PARAMETRY]
    img_size = (256, 256)

    # Wycięcie fragmentu zdjęcia
    (x, y, w, h) = face
    face_area = image[y:y + h, x:x + w]

    (x, y, w, h) = flash_image_face
    flash_face_area = flash_image[y:y + h, x:x + w]

    # Przetworzenie obrazu - sam obszar twarzy
    face_area = cv2.resize(face_area, img_size)
    flash_face_area = cv2.resize(flash_face_area, img_size)

    face_area = cv2.cvtColor(face_area, cv2.COLOR_BGR2Luv)
    flash_face_area = cv2.cvtColor(flash_face_area, cv2.COLOR_BGR2Luv)

    # Obliczenie średniej wartości kanału L
    face_area_L = np.mean(face_area[:, :, 0])
    flash_face_area_L = np.mean(flash_face_area[:, :, 0])

    diff = cv2.absdiff(face_area_L, flash_face_area_L)

    mean_diff = diff.mean()

    print("Mean diff dla obszaru twarzy: ", mean_diff)
    
    # Sprawdzenie zdjęcia bez twarzy
    # 1. Resize, 2. Obliczenie współczynnika proporcji, 3. Obliczenie średniej "ręcznie" dla obszaru poza twarzą
    img_size = (512, 512)
    image_no_face = image.copy()
    image_no_face = cv2.resize(image_no_face, img_size)
    image_no_face = cv2.cvtColor(image_no_face, cv2.COLOR_BGR2Luv)
    flash_image_no_face = flash_image.copy()
    flash_image_no_face = cv2.resize(flash_image_no_face, img_size)
    flash_image_no_face = cv2.cvtColor(flash_image_no_face, cv2.COLOR_BGR2Luv)
    
    proportion_image = (image.shape[0] / img_size[0], image.shape[1] / img_size[1])
    proportion_flash_image = (flash_image.shape[0] / img_size[0], flash_image.shape[1] / img_size[1])
    
    face_position = (face[0] / proportion_image[0], face[1] / proportion_image[1], face[2] / proportion_image[0], face[3] / proportion_image[1])
    flash_face_position = (flash_image_face[0] / proportion_flash_image[0], flash_image_face[1] / proportion_flash_image[1], flash_image_face[2] / proportion_flash_image[0], flash_image_face[3] / proportion_flash_image[1])
    
    # Obliczenie obszaru poza twarzą
    mean_noflash = 0
    count_noflash = 0
    mean_flash = 0
    count_flash = 0
    for i in range (0, 512):
        for j in range(0, 512):
            if (i >= face_position[1] and i <= face_position[1] + face_position[3] and j >= face_position[0] and j <= face_position[0] + face_position[2]):
                continue
            else:
                mean_noflash += image_no_face[i, j, 0]
                count_noflash += 1
            
            if (i >= flash_face_position[1] and i <= flash_face_position[1] + flash_face_position[3] and j >= flash_face_position[0] and j <= flash_face_position[0] + flash_face_position[2]):
                continue
            else:
                mean_flash += flash_image_no_face[i, j, 0]
                count_flash += 1
                
    mean_noflash /= count_noflash
    mean_flash /= count_flash
    
    diff = cv2.absdiff(mean_noflash, mean_flash)
    mean_diff = diff.mean()
    
    print("Mean diff dla zdjęcia z wyciętym obszarem twarzy: ", mean_diff)

In [3]:
# Testowanie funkcji
img_auth_nolight = cv2.imread("../../../Dane/Lighting/Original_daylight_nolight.png")
img_auth_light = cv2.imread("../../../Dane/Lighting/Original_daylight_light.png")
img_spoof_nolight = cv2.imread("../../../Dane/Lighting/Spoof_daylight_nolight.png")
img_spoof_light = cv2.imread("../../../Dane/Lighting/Spoof_daylight_light.png")

# Wykrywanie twarzy
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    return faces[0]

face = detect_face(img_auth_nolight)
flash_face = detect_face(img_auth_light)

analyze_lighting(img_auth_nolight, face, img_auth_light, flash_face)

In [4]:
face = detect_face(img_spoof_nolight)
flash_face = detect_face(img_spoof_light)

analyze_lighting(img_spoof_nolight, face, img_spoof_light, flash_face)